<a href="https://colab.research.google.com/github/camis08/IA_UC9/blob/main/aula_02_agente_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 02 — Interpretando mensagens com um modelo aberto e gratuito

**UC9 — Compreender e Aplicar Machine Learning em soluções de IA**  
**Projeto integrador:** Agentes de IA para o setor imobiliário  
**Duração:** 3 horas presenciais

## O que construiremos

Nesta aula, criaremos a primeira parte de um agente imobiliário capaz de:

1. receber uma mensagem escrita normalmente por um cliente;
2. usar um modelo de linguagem aberto para interpretar a mensagem;
3. transformar o texto em dados estruturados;
4. validar os dados com Pydantic;
5. usar regras Python para escolher o próximo passo.

### Exemplo de entrada

> Quero comprar um apartamento de dois quartos em Águas Claras. Posso pagar até R$ 450 mil e talvez precise financiar.

### Resultado esperado

```json
{
  "finalidade": "compra",
  "tipo_imovel": "apartamento",
  "regiao_desejada": "Águas Claras",
  "orcamento_maximo": 450000,
  "quantidade_quartos": 2,
  "precisa_financiamento": true
}
```

> **Não utilizaremos OpenAI API, cartão, créditos ou chave paga.**  
> O modelo será baixado e executado dentro do Google Colab.


## Objetivos de aprendizagem

Ao final da aula, você deverá conseguir:

- explicar a diferença entre modelo de linguagem e agente;
- carregar um modelo aberto com a biblioteca Transformers;
- enviar mensagens nos papéis `system` e `user`;
- gerar uma resposta localmente, sem API comercial;
- orientar o modelo a devolver JSON;
- extrair o JSON da resposta;
- validar o resultado com Pydantic;
- separar a interpretação feita pela IA das regras determinísticas em Python;
- testar o protótipo com mensagens diferentes.


## Fluxo resumido em seis passos:

1) Carrega o tokenizer → responsável por converter texto em tokens e vice-versa.

2) Carrega o modelo → baixa os pesos do LLM e o prepara para uso.
3) Formata a conversa → organiza as mensagens no formato esperado pelo modelo (chat template).
4) Tokeniza a entrada → transforma o texto em tensores numéricos e os envia para CPU ou GPU.
5) Gera novos tokens → o modelo prevê, um a um, os próximos tokens mais prováveis até atingir o limite ou encontrar o fim da resposta.
6) Decodifica os tokens → converte os novos tokens novamente em texto legível e retorna a resposta ao usuário.

# 1. Preparação do Google Colab

Antes de executar o notebook:

1. Abra **Ambiente de execução**.
2. Escolha **Alterar tipo de ambiente de execução**.
3. Em acelerador de hardware, escolha uma GPU disponível, preferencialmente **T4**.
4. Salve a configuração.

O notebook também possui uma alternativa menor para execução sem GPU, mas ela será mais lenta e poderá interpretar as mensagens com menor qualidade.


In [1]:
# Instalação das bibliotecas utilizadas na aula.


!pip install -q -U "transformers>=4.45.0" "accelerate>=0.34.0" "pydantic>=2.8.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 60.7 MB/s eta 0:00:00


# 2. Verificando o ambiente

A célula seguinte identifica se existe uma GPU disponível.

Usaremos:

- **Qwen2.5 1.5B Instruct** quando houver GPU;
- **Qwen2.5 0.5B Instruct** como alternativa mais leve quando houver apenas CPU.

Os dois modelos são públicos e não exigem chave de API.


In [2]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPU disponível?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: o modelo será executado pela CPU e poderá ficar mais lento.")


Python: 3.12.13
PyTorch: 2.11.0+cu128
GPU disponível? True
GPU: Tesla T4


In [3]:
# Escolha automática do modelo conforme o hardware disponível.

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"
MODELO_CPU = "Qwen/Qwen2.5-0.5B-Instruct"

MODELO_ID = MODELO_GPU if torch.cuda.is_available() else MODELO_CPU

print("Modelo selecionado:", MODELO_ID)


Modelo selecionado: Qwen/Qwen2.5-1.5B-Instruct


# 3. Carregando o modelo aberto

Na primeira execução, os arquivos do modelo serão baixados para o ambiente do Colab.

Isso pode levar alguns minutos. Depois de carregado, o texto será processado no próprio ambiente, sem cobrança por mensagem.

### Conceitos importantes

- **Tokenizer:** transforma texto em números que o modelo entende.
- **Modelo:** recebe os tokens e prevê os próximos tokens.
- **Pesos:** parâmetros aprendidos durante o treinamento.
- **Instruct:** versão ajustada para seguir instruções e conversar.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

#carregando modelo
#baixa o modelo do carinha feliz(hugging face) e carrega os bilhões de parâmetros de memória

model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype= "auto",
    device_map="auto",
    low_cpu_mem_usage=True,
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
model.eval()
#coloca no mode interferência e deixa o mais eficiente

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [6]:
dispositivo_modelo = next(model.parameters()).device
print("Dispositivo do modelo: ", dispositivo_modelo )

Dispositivo do modelo:  cuda:0


# 4. Criando uma função para conversar com o modelo

Modelos de chat esperam mensagens com papéis:

- `system`: regras e comportamento;
- `user`: solicitação do usuário;
- `assistant`: respostas anteriores, quando houver histórico.

A função abaixo aplica o formato de conversa correto do modelo e gera somente a continuação produzida pelo assistente.


In [7]:
mensagens = [
    {
        "role" : "user",
        "content" : "Quem descobriu o Brasil?",
    }
]

In [8]:
def gerar_resposta(
    mensagens: list[dict[str, str]],
    max_novos_tokens: int = 300,
) -> str:
    """Gera uma resposta usando o modelo aberto carregado no notebook."""

    #Transformando a mensagem no formato esperado:
    texto_formatado = tokenizer.apply_chat_template( # o apply_chat_template monta o formato automaticamente
        mensagens,
        tokenize=False,
        add_generation_prompt=True, #Coloca no final da converso o "Assistant:"
    )
    #print(texto_formatado)

    #Tokenizando -> Transformar texto em números
    entradas = tokenizer(
        texto_formatado,
        return_tensors="pt" #Retorna tensores do PyTorch
    )
    #print(entradas)

    #Envia cada item da entrada para onde o modelo está rodando
    entradas = {
        nome: tensor.to(dispositivo_modelo)
        for nome, tensor in entradas.items()
    }

    #Entrando no modo inferência do pytorch:
    with torch.inference_mode():
      #É a parte mais importante, é onde a IA começa aa gerar a resposta, prevendo um token por vez
      saidas = model.generate(
          **entradas, #Os ** significa: desempacotar o dict
          max_new_tokens=max_novos_tokens, #Limita a quantidade máxima de tokens que poderão ser gerados. Evita resposta grandes ou infinitas
          do_sample=False, # Sempre escolhe a palavra mais provável. O True, deixa ele mais criativo
          repetition_penalty=1.05, #Prevenir repeticoes de palavras.
          pad_token_id=tokenizer.eos_token_id, #Define qual é o token que será utilizado para preencher espaços quando necessário.
      )

    quantidade_tokens_entrada = entradas["input_ids"].shape[1] #Conta quantos tokens existiam originalmente.
    novos_tokens = saidas[0][quantidade_tokens_entrada:] #Aqui serve para descartarmos a pergunta. Ficamos apenas com as resposta.

    #Converter para texto:
    return tokenizer.decode(novos_tokens, skip_special_tokens=True).strip() # Remove tokens especiais, elimina espaços em branco no inicio o no final da resposta.


## Primeiro teste

Neste momento, ainda não estamos construindo um agente completo. Estamos verificando se o modelo consegue receber e interpretar uma mensagem em português.


In [9]:
mensagens_teste = [
    {
        "role": "system",
        "content": (
            "Você é um assistente imobiliário. "
            "Responda português, de forma curta e objetiva."
        ),
    },
    {
        "role": "user",
        "content": (
            "Quero comprar um apartamento de dois quartos em Águas Claras, "
            "até R$ 450 mil. O que você entendeu?"
        ),
    },
]

In [10]:
resposta_teste = gerar_resposta(mensagens_teste, max_novos_tokens=150)

In [11]:
resposta_teste

'Entendi que você está procurando por um apartamento com duas camas em Águas Claras, com valor máximo de R$ 450 mil.'

# 5. Definindo a estrutura esperada com Pydantic

Não queremos que o restante do sistema dependa de uma resposta em texto livre.

O modelo de linguagem ficará responsável por **interpretar a linguagem humana**.  
O Pydantic ficará responsável por **validar a estrutura e os tipos dos dados**.

Por exemplo:

- orçamento deve ser número ou `null`;
- quantidade de quartos deve ser número natural ou `null`;
- finalidade deve ter apenas valores previamente permitidos;
- campos extras não devem ser aceitos.


In [12]:
from typing import Literal  #O Literal permite limitar um campo a valores específicos.
from pydantic import BaseModel, ValidationError, Field, ConfigDict, field_validator

#ConfigDict: configurar comportamento do modelo Pydanct
#Field: adicionar regras extras em um campo
#fiel_validator: permite criar validações personalizadas para campos específicos
#ValidationError: isar erros dentro do Execpt

In [13]:
class QualificacaoCliente(BaseModel):#estruta à partir da msg do cliente
    model_config = ConfigDict(extra = "forbid")#não aceita campos não definidos na classe
    #estrutura rigorosa e prevível

    finalidade: Literal['compra', 'locacao', 'nao_informado']
    # 'nao_informado' eh o None

    tipo_imovel: str | None = None
    regiao_desejada: str | None = None
    orcamento_maximo: float | None = None
    quantidade_quartos: int | None = None
    precisa_financiamento: bool | None = None
    # aceita True, False or None -> se o cliente não informar é None
    quantidade_vagas: int | None = None
    assunto_sensivel: bool = False
    resumo: str = Field(max_length=200)

    @field_validator("orcamento_maximo")
    #Decorator: cria validação específica para o campo
    #Diz para o Pydantic -> ao validar o campo, executa essa função
    @classmethod
    #indica que o método pertence a classe
    def validar_orcamento(cls, valor: float | None) -> float | None:
        if valor is not None and valor <= 0:
            raise ValueError("Orçamento deve ser maior que zero.")
        return valor


    @field_validator("quantidade_quartos")
    @classmethod
    def validar_quantidade_quartos(cls, valor: int | None) -> int | None:
        if valor is not None and valor <= 0: #tomar cuidado qnd tipo_imovel ≠ [casa,ap]
            raise ValueError("Quantidade de quartos deve ser maior que zero.")
        return valor

    @field_validator("quantidade_vagas")
    @classmethod
    def validar_quantidade_vagas(cls, valor: int | None) -> int | None:
        if valor is not None and valor <= 0:
            raise ValueError("Quantidade de vagas deve ser maior que zero.")
        return valor

    @field_validator("assunto_sensivel",  mode="before")
    @classmethod
    def preencher_assunto_sensivel(cls, valor: bool | None) -> bool:
        if valor is None:
            return False
        return valor


In [14]:
QualificacaoCliente.model_json_schema()

{'additionalProperties': False,
 'properties': {'finalidade': {'enum': ['compra', 'locacao', 'nao_informado'],
   'title': 'Finalidade',
   'type': 'string'},
  'tipo_imovel': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Tipo Imovel'},
  'regiao_desejada': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Regiao Desejada'},
  'orcamento_maximo': {'anyOf': [{'type': 'number'}, {'type': 'null'}],
   'default': None,
   'title': 'Orcamento Maximo'},
  'quantidade_quartos': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
   'default': None,
   'title': 'Quantidade Quartos'},
  'precisa_financiamento': {'anyOf': [{'type': 'boolean'}, {'type': 'null'}],
   'default': None,
   'title': 'Precisa Financiamento'},
  'quantidade_vagas': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
   'default': None,
   'title': 'Quantidade Vagas'},
  'assunto_sensivel': {'default': False,
   'title': 'Assunto Sensivel',
   'type': 'b

## Testando o Pydantic antes de usar a IA

É importante compreender a validação separadamente.


In [15]:
dados_validos = {
    "finalidade": "compra",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": 450000,
    "quantidade_quartos": 2,
    "quantidade_vagas": 2,
    "precisa_financiamento": True,
    "assunto_sensivel": False,
    "resumo": "Cliente procura apartamento de 2 quartos em Águas Claras."
}

In [16]:
cliente_validado = QualificacaoCliente.model_validate(dados_validos) # recebe um dic. e tenta criar um obj.(qualif. cliente)

In [17]:
print(cliente_validado.model_dump_json(indent=2))

{
  "finalidade": "compra",
  "tipo_imovel": "apartamento",
  "regiao_desejada": "Águas Claras",
  "orcamento_maximo": 450000.0,
  "quantidade_quartos": 2,
  "precisa_financiamento": true,
  "quantidade_vagas": 2,
  "assunto_sensivel": false,
  "resumo": "Cliente procura apartamento de 2 quartos em Águas Claras."
}


In [18]:
cliente_validado.finalidade

'compra'

In [19]:
cliente_validado.quantidade_quartos

2

In [20]:
cliente_validado.orcamento_maximo

450000.0

Durante esse processo, o Pydantic verifica:

- presença dos campos obrigatórios;
- tipos dos dados;
- valores aceitos pelo Literal;
- campos adicionais;
- tamanho do resumo;
- validações personalizadas.

Caso tudo esteja certo, é criado um objeto:

Esse objeto [cliente_validado] não é mais apenas um dicionário. Ele é uma instância da classe QualificacaoCliente.

Poderíamos acessar seus atributos: print(cliente_validado.finalidade)

In [21]:
# Exemplo propositalmente inválido.

dados_invalidos = {
    "finalidade": "troca",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Águas Claras",
    "orcamento_maximo": -100,
    "quantidade_quartos": 0,
    "precisa_financiamento": "talvez",
    "assunto_sensivel": False,
    "resumo": "Teste"
}



In [22]:
try:
    cliente_invalido = QualificacaoCliente.model_validate(dados_invalidos)
except ValidationError as e:
    print(e)

4 validation errors for QualificacaoCliente
finalidade
  Input should be 'compra', 'locacao' or 'nao_informado' [type=literal_error, input_value='troca', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
orcamento_maximo
  Value error, Orçamento deve ser maior que zero. [type=value_error, input_value=-100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
quantidade_quartos
  Value error, Quantidade de quartos deve ser maior que zero. [type=value_error, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
precisa_financiamento
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value='talvez', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_parsing


# 6. Orientando o modelo a produzir JSON

O modelo deve receber uma instrução muito clara:

- devolver somente JSON;
- usar exatamente os nomes dos campos;
- não inventar dados;
- usar `null` quando a informação estiver ausente;
- converter valores monetários para número;
- usar valores padronizados para a finalidade.

Mesmo assim, o modelo pode errar. Por isso, ainda precisaremos extrair e validar o JSON.


In [23]:
INSTRUCAO_EXTRACAO = '''
Você é um componente de extração de informações de uma imobiliária.

Analise a mensagem do cliente e responda SOMENTE com um objeto JSON válido.
Não use Markdown, não use ``` e não escreva explicações fora do JSON.

Use exatamente estas chaves:
{
  "finalidade": "compra | locacao | nao_informado",
  "tipo_imovel": "texto ou null",
  "regiao_desejada": "texto ou null",
  "orcamento_maximo": "número ou null",
  "quantidade_vagas": "inteiro ou null",
  "quantidade_quartos": "inteiro ou null",
  "precisa_financiamento": "true, false ou null",
  "assunto_sensivel": "true ou false",
  "resumo": "resumo o que o cliente procura português e informe se irá precisar ou não de financiamento"
}

Regras:
1. Não invente informações.
2. Use null quando a informação não estiver presente.
3. Converta expressões como "450 mil" para 450000.
4. Use "locacao", sem cedilha ou acento, no campo finalidade.
5. Marque assunto_sensivel como true quando o cliente pedir orientação
   jurídica, contratual ou uma análise financeira detalhada.
6. Dúvida simples sobre necessidade de financiamento não é, sozinha,
   uma orientação financeira detalhada.
7. É obrigatório que na resposta bruta ou reusmo diga se irá precisar ou não de financiamento"
'''


In [24]:
mensagem_cliente = (
    "Quero alugar uma casa no Lago Sul, com no mínimo três vagas de garagem, "
    "até R$50 mil. Não irei precisar de financiamento."
)

In [25]:
mensagem_extracao = [
    {
        "role": "system", "content": INSTRUCAO_EXTRACAO},
    {
        "role": "user", "content": mensagem_cliente}
]

In [26]:
resposta_bruta = gerar_resposta(mensagem_extracao)

In [27]:
print(resposta_bruta)

{
  "finalidade": "locacao",
  "tipo_imovel": "casa",
  "regiao_desejada": "Lago Sul",
  "orcamento_maximo": 50000,
  "quantidade_vagas": 3,
  "quantidade_quartos": null,
  "precisa_financiamento": false,
  "assunto_sensivel": false,
  "resumo": "Aluguel de casa no Lago Sul com três vagas de garagem e orçamento máximo de R$50 mil. Não preciso de financiamento."
}


# 7. Extraindo o objeto JSON da resposta

Embora a instrução peça somente JSON, um modelo pequeno pode acrescentar algum texto.

A função seguinte procura o primeiro objeto JSON válido dentro da resposta.


In [28]:
import json

def extrair_json(texto: str) -> dict:
    texto_limpo = (
        texto.strip()
        .replace("```json","")
        .replace("```JSON","")
        .replace("```","")
        .strip()
    )

    inicio = texto_limpo.find("{")
    if inicio == -1:
        raise ValueError("Nenhum objeto JSON encontrado")

    decodificador = json.JSONDecoder()
    objeto, _ = decodificador.raw_decode(texto_limpo[inicio:])

    if not isinstance(objeto, dict):
        raise ValueError("Objeto JSON inválido")

    return objeto

In [29]:
dados_extraidos = extrair_json(resposta_bruta)

In [30]:
dados_extraidos

{'finalidade': 'locacao',
 'tipo_imovel': 'casa',
 'regiao_desejada': 'Lago Sul',
 'orcamento_maximo': 50000,
 'quantidade_vagas': 3,
 'quantidade_quartos': None,
 'precisa_financiamento': False,
 'assunto_sensivel': False,
 'resumo': 'Aluguel de casa no Lago Sul com três vagas de garagem e orçamento máximo de R$50 mil. Não preciso de financiamento.'}

# 8. Validando a interpretação

Agora temos três camadas distintas:

```text
Mensagem do cliente
        ↓
Modelo aberto interpreta
        ↓
Função extrai o JSON
        ↓
Pydantic valida os dados
```

Se alguma camada falhar, o sistema deve mostrar um erro controlado em vez de continuar com dados incorretos.


In [31]:
try:
    dado_validado = QualificacaoCliente.model_validate(dados_extraidos)
    print(dado_validado.model_dump_json(indent=2))
except ValidationError as e:
    print(e)

{
  "finalidade": "locacao",
  "tipo_imovel": "casa",
  "regiao_desejada": "Lago Sul",
  "orcamento_maximo": 50000.0,
  "quantidade_quartos": null,
  "precisa_financiamento": false,
  "quantidade_vagas": 3,
  "assunto_sensivel": false,
  "resumo": "Aluguel de casa no Lago Sul com três vagas de garagem e orçamento máximo de R$50 mil. Não preciso de financiamento."
}


# 9. Criando uma função completa de interpretação

Esta função reúne:

1. montagem das mensagens;
2. geração local;
3. extração do JSON;
4. validação com Pydantic;
5. tratamento dos erros mais comuns.


In [32]:
def interpretar_mensagem_cliente(
    mensagem: str
) -> tuple[QualificacaoCliente | None, str]:

    """
    Interpreta msg do cliente
    Retorno:
    - um obj QualificacaoCliente se válido;
    - a resposta bruta do modelo para análise
    """

    mensagens = [
        {"role": "system","content": INSTRUCAO_EXTRACAO},
        {"role": "user","content": mensagem}
    ]

    resposta = gerar_resposta(mensagens)

    try:
        objeto_json = extrair_json(resposta)
        qualificacao = QualificacaoCliente.model_validate(objeto_json)
        return qualificacao, resposta
    except (ValueError, json.JSONDecodeError ,ValidationError) as e:
        print("Nçao foi possível validar resposta do modelo")
        print("Tipo de erro", type(e).__name__)
        print("Mensagem de erro", e)
        return None, resposta

In [33]:
mensagem = (
    "Quero alugar uma casa no Lago Sul com 4 quartos, com no mínimo três vagas de garagem. "
    "Consigo pagar até R$20 mil por mês."
    "Não irei precisar de financiamento."
)

In [34]:
resultado, resposta_modelo = interpretar_mensagem_cliente(mensagem)

In [35]:
#print(resultado)

In [36]:
#print(resposta_modelo)

In [37]:
if resultado:
    print(resultado.model_dump_json(indent=2))
else:
    print("A resposta do modelo não foi validada")
    print(resposta_modelo)

{
  "finalidade": "locacao",
  "tipo_imovel": "casa",
  "regiao_desejada": "Lago Sul",
  "orcamento_maximo": 200000.0,
  "quantidade_quartos": 4,
  "precisa_financiamento": false,
  "quantidade_vagas": 3,
  "assunto_sensivel": false,
  "resumo": "Aluguel de casa no Lago Sul com 4 quartos, 3 vagas de garagem e orçamento máximo de R$20 mil por mês."
}


# 10. O modelo interpreta; o Python decide

Não devemos entregar todas as decisões ao modelo de linguagem.

Uma arquitetura mais segura é:

- **modelo aberto:** interpreta a mensagem;
- **Pydantic:** valida os dados;
- **Python:** aplica as regras da imobiliária;
- **humano:** assume situações sensíveis ou de maior responsabilidade.

Nesta aula, o Python escolherá entre três ações:

1. solicitar uma informação ausente;
2. consultar imóveis;
3. encaminhar para um corretor humano.


In [38]:
CAMPOS_NECESSARIOS = [
    "finalidade",
    "tipo_imovel",
    "regiao_desejada",
    "orcamento_maximo"
]

In [60]:
def decidir_proxima_acao(
    cliente: QualificacaoCliente
    ) -> dict:

    """
    Aplica regras determinadas após
    interpretação de modelo
    """

    if cliente.assunto_sensivel:
        return {
            "acao" : "encaminhar humano",
            "mensagem" : ("Essa solicitação precisa ser analisada por um Corretor!")
        }

    dados = cliente.model_dump()

    dados_ausentes = []
    for campo in CAMPOS_NECESSARIOS:
        valor = dados.get(campo)
        if campo == "finalidade":
            ausente = valor == "nao_informado"
        else:
            ausente = valor is None
            # valor é falso

        if ausente:
            dados_ausentes.append(campo)


    if dados_ausentes:
        primeiro_campo = dados_ausentes[0]

        perguntas =  {
            "finalidade": "Pretende comprar ou alugar um imóvel?",
            "tipo_imovel": "Qual tipo de imóvel procura?",
            "regiao_desejada": "Possui preferência de região?",
            "orcamento_maximo": "Informe seu orçamento máximo",
        }
        return {
            "acao": "perguntar_dado",
            "campo": primeiro_campo,
            "pergunta": perguntas[primeiro_campo]
            }

    return {
        "acao": "consultar_imoveis",
        "filtros": {
            "finalidade": cliente.finalidade,
            "tipo_imovel": cliente.tipo_imovel,
            "regiao_desejada": cliente.regiao_desejada,
            "orcamento_maximo": cliente.orcamento_maximo,
            "quantidade_quartos": cliente.quantidade_quartos,
            "quantidade_vagas": cliente.quantidade_vagas,
            "precisa_financiamento": cliente.precisa_financiamento,
        },
            "mensagem": "Dados mínimos coletados, podemos consultar os imóveis"
        }

In [61]:
resultado

QualificacaoCliente(finalidade='locacao', tipo_imovel='casa', regiao_desejada='Lago Sul', orcamento_maximo=200000.0, quantidade_quartos=4, precisa_financiamento=False, quantidade_vagas=3, assunto_sensivel=False, resumo='Aluguel de casa no Lago Sul com 4 quartos, 3 vagas de garagem e orçamento máximo de R$20 mil por mês.')

In [62]:
if resultado:
    decisao = decidir_proxima_acao(resultado)
    print(decisao)

{'acao': 'consultar_imoveis', 'filtros': {'finalidade': 'locacao', 'tipo_imovel': 'casa', 'regiao_desejada': 'Lago Sul', 'orcamento_maximo': 200000.0, 'quantidade_quartos': 4, 'quantidade_vagas': 3, 'precisa_financiamento': False}, 'mensagem': 'Dados mínimos coletados, podemos consultar os imóveis'}


In [63]:
print(json.dumps(decisao, indent=2))

{
  "acao": "consultar_imoveis",
  "filtros": {
    "finalidade": "locacao",
    "tipo_imovel": "casa",
    "regiao_desejada": "Lago Sul",
    "orcamento_maximo": 200000.0,
    "quantidade_quartos": 4,
    "quantidade_vagas": 3,
    "precisa_financiamento": false
  },
  "mensagem": "Dados m\u00ednimos coletados, podemos consultar os im\u00f3veis"
}


# 11. Testando mensagens diferentes

Avalie se o modelo:

- extrai corretamente o tipo de imóvel;
- interpreta valores como “meio milhão”;
- usa `null` quando faltam dados;
- diferencia compra de locação;
- detecta assuntos que exigem intervenção humana.

> Modelos pequenos podem errar. O objetivo não é esconder os erros, mas aprender a identificá-los e controlá-los.


In [64]:
cenarios = [
    "Quero alugar um apartamento com dois quartos no Plano Piloto, até R$5 mil"
    "Pode ser no Jardim Botânico também"
    "Precisa ter uma vaga de garagem"
    "Pode ter pet?"
    "Possui um imóvel com o que quero?"
    "Analise esse contrato e diga se existe alguma cláusula ilegal"
]

In [72]:
for numero, cenario in enumerate(cenarios, start=1):
    print("\n"+"="*80)
    print(f"Cenario {numero}")
    print("Cliente", cenario)

    cliente, resposta_modelo = interpretar_mensagem_cliente(cenario)

    if cliente is None:
        print("\n Resposta não validade")
        print(resposta_modelo)

    print("\n Interpretação validada")
    print(cliente.model_dump_json(indent=2))

    print("\n Próxima ação:")
    print(json.dumps(
        decidir_proxima_acao(cliente),\
        indent=2)
    )


Cenario 1
Cliente Quero alugar um apartamento com dois quartos no Plano Piloto, até R$5 milPode ser no Jardim Botânico tambémPrecisa ter uma vaga de garagemPode ter pet?Possui um imóvel com o que quero?Analise esse contrato e diga se existe alguma cláusula ilegal

 Interpretação validada
{
  "finalidade": "locacao",
  "tipo_imovel": "apartamento",
  "regiao_desejada": "Plano Piloto",
  "orcamento_maximo": 50000.0,
  "quantidade_quartos": 2,
  "precisa_financiamento": false,
  "quantidade_vagas": 1,
  "assunto_sensivel": false,
  "resumo": "O cliente procura um apartamento com dois quartos no Plano Piloto, até R$5 mil, com uma vaga de garagem e sem necessidade de financiamento."
}

 Próxima ação:
{
  "acao": "consultar_imoveis",
  "filtros": {
    "finalidade": "locacao",
    "tipo_imovel": "apartamento",
    "regiao_desejada": "Plano Piloto",
    "orcamento_maximo": 50000.0,
    "quantidade_quartos": 2,
    "quantidade_vagas": 1,
    "precisa_financiamento": false
  },
  "mensagem": "

# 12. Registrando os resultados dos testes

Agentes não devem ser avaliados apenas pela impressão de que “parecem funcionar”.

Crie casos de teste e compare:

- resultado esperado;
- resultado produzido;
- erro encontrado;
- possível ajuste na instrução ou nas regras.


# Atividade prática — Parte 1

Em dupla ou trio:

1. Execute o notebook até a função `interpretar_mensagem_cliente`.
2. Crie pelo menos cinco mensagens de clientes.
3. Inclua formas diferentes de informar o orçamento:
   - `450 mil`;
   - `R$ 450.000`;
   - `meio milhão`;
   - `até uns 500k`.
4. Registre os erros encontrados.
5. Ajuste a instrução para tentar reduzir os erros.


In [ ]:
# Escreva aqui os cenários criados pelo grupo.

cenarios_do_grupo = [
    "",
    "",
    "",
    "",
    "",
]

for cenario in cenarios_do_grupo:
    if not cenario.strip():
        continue

    cliente, resposta_modelo = interpretar_mensagem_cliente(cenario)

    print("\nMensagem:", cenario)
    print("Resposta bruta:", resposta_modelo)

    if cliente:
        print("Validado:", cliente.model_dump())
        print("Ação:", decidir_proxima_acao(cliente))


# Atividade prática — Parte 2

Modifique o protótipo para extrair mais duas informações relevantes para o projeto imobiliário.

Sugestões:

- necessidade de vaga de garagem;
- prazo para mudança;
- preferência por imóvel mobiliado;
- área mínima;
- possui animal de estimação;
- aceita contato por WhatsApp;
- possui imóvel para vender;
- urgência do atendimento.

Você deverá:

1. alterar a classe Pydantic;
2. alterar a instrução;
3. testar pelo menos três mensagens;
4. explicar um erro cometido pelo modelo;
5. mostrar como o Pydantic ou uma regra Python ajuda a controlar esse erro.


In [ ]:
# ESPAÇO PARA A NOVA VERSÃO DO GRUPO

# 1. Crie uma nova classe, por exemplo: QualificacaoClienteV2
# 2. Crie uma nova instrução.
# 3. Crie uma nova função de interpretação.
# 4. Execute os testes.


# Desafio opcional — Tentativa de correção automática

Quando a resposta vier inválida, podemos enviar ao modelo:

- a resposta anterior;
- o erro de validação;
- a instrução para corrigir apenas o JSON.

Esse mecanismo será estudado com mais profundidade quando trabalharmos o ciclo de execução do agente.

Pseudocódigo:

```python
resposta = gerar()

try:
    validar(resposta)
except Erro:
    nova_resposta = gerar(
        "Corrija o JSON anterior conforme este erro..."
    )
```


# O que construímos

Ao final desta aula, o protótipo já consegue:

```text
Mensagem em português
        ↓
Modelo aberto executado no Colab
        ↓
Extração de informações
        ↓
JSON
        ↓
Validação com Pydantic
        ↓
Regras Python
        ↓
Perguntar, consultar ou encaminhar
```

## O que ainda não construímos

O protótipo ainda não executa ferramentas automaticamente.

Na próxima aula, acrescentaremos funções como:

- consultar uma base fictícia de imóveis;
- calcular valores;
- devolver o resultado da ferramenta ao modelo;
- controlar o número máximo de etapas do agente.


# Checklist de entrega

A dupla ou trio deverá apresentar:

- [ ] modelo aberto carregado sem API comercial;
- [ ] pelo menos uma mensagem interpretada;
- [ ] resposta em JSON;
- [ ] validação com Pydantic;
- [ ] regra Python para escolher a próxima ação;
- [ ] cinco cenários de teste;
- [ ] pelo menos um erro identificado;
- [ ] uma melhoria realizada na instrução ou no modelo de dados;
- [ ] explicação da diferença entre modelo e agente.

## Conclusão

O protótipo não é apenas um conjunto de respostas prontas: um modelo de linguagem aberto está interpretando mensagens reais. Entretanto, a IA não deve controlar tudo. A combinação entre modelo, validação, regras e supervisão humana torna a solução mais previsível e segura.
